# BNY Corporate-Actions EDA

**Question:** How well can *public* corporate-action documents populate the fields required in a real BNY client notification?

**Canonical target:** BNY notification schema (`src/eda/bny_schema.py`)

**Public corpus (this run):** SEC EDGAR tender-related filings via EFTS — `SC TO-T`, `SC TO-T/A`, `SC TO-I`, `SC TO-I/A`, `SC 14D9`, `SC 14D9/A` (2020–2025).

### Evidence discipline
| Label | Meaning |
|-------|---------|
| **Observed** | Present in EFTS metadata or successful document download/parse |
| **Inferred** | Regex mention detection / heuristic derivation |
| **Internal** | Definitionally unavailable from public documents |
| **N/A** | Not relevant for an event type (assumption, labeled) |

Missing scalars → conceptually `null`. Missing lists (e.g. `available_options`) → conceptually `[]`.  
`notification_type` is kept **separate** from `corporate_action_type`.


In [ ]:
from pathlib import Path
import sys, warnings
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 120)
sns.set_theme(style='whitegrid')

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from eda.corpus import load_corpus
from eda.overview import summarize_dataset, overview_tables, plot_form_distribution, plot_temporal_volume
from eda.events import analyze_events, plot_event_distributions
from eda.documents import analyze_documents, plot_format_distribution, sample_documents_for_content
from eda.schema_coverage import analyze_schema_coverage
from eda.amendments import analyze_amendments
from eda.quality import analyze_data_quality
from eda.missingness import classify_missingness, ground_truth_recommendations
from eda.mvp import recommend_mvp_design
from eda.bny_schema import BNY_SCHEMA
from eda.config import PROCESSED_DIR, FIGURES_DIR

OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)
(OUTPUTS / 'figures').mkdir(exist_ok=True)
print('fields in BNY schema:', len(BNY_SCHEMA))


## 1. Dataset overview

In [ ]:
docs, events = load_corpus()
overview = summarize_dataset(docs, events)
tables = overview_tables(docs, events)
display(tables['summary'])
display(tables['form_counts'])
display(tables['family_counts'])
print(overview['source_note'])
print(f"Events with multiple documents: {(events['n_documents']>1).sum()} ({100*(events['n_documents']>1).mean():.1f}%)")
print(f"Events with amendments: {int(events['has_amendment'].sum())} ({overview['pct_events_with_amendments']}%)")


In [ ]:
fig = plot_form_distribution(docs); plt.show()
fig = plot_temporal_volume(docs); plt.show()
fig = plot_event_distributions(events); plt.show()


In [ ]:
event_stats = analyze_events(docs, events)
display(event_stats['unusual_document_volume'].head(10))
display(event_stats['unusual_amendment_volume'].head(10))
display(event_stats['repeated_issuers'].head(15))
print(event_stats['note'])


## 2. BNY schema coverage analysis

Text-field percentages are **inferred** from regex presence on a stratified download sample and are **not** extraction accuracy.
Internal fields are definitionally 0% public coverage.


In [ ]:
sample_path = PROCESSED_DIR / 'content_sample.parquet'
if sample_path.exists():
    content_sample = pd.read_parquet(sample_path)
else:
    content_sample = sample_documents_for_content(docs, n_per_form=8)
    content_sample.to_parquet(sample_path, index=False)
print(content_sample.groupby(['form','status']).size())
coverage = analyze_schema_coverage(docs, events, content_sample)
print(coverage['note'])
display(coverage['schema_coverage'][[
    'field','public_coverage_pct','missing_pct','primary_source',
    'explicit_or_derived','difficulty','evidence_type','denominator_note'
]])
coverage['schema_coverage'].to_csv(OUTPUTS / 'schema_coverage.csv', index=False)


In [ ]:
cov = coverage['schema_coverage'].dropna(subset=['public_coverage_pct']).sort_values('public_coverage_pct')
fig, ax = plt.subplots(figsize=(10,10))
sns.barplot(data=cov, y='field', x='public_coverage_pct', hue='group', dodge=False, ax=ax)
ax.set_xlim(0,100); ax.set_title('BNY schema public coverage')
fig.tight_layout(); fig.savefig(OUTPUTS/'figures'/'schema_coverage.png', dpi=150); plt.show()


## 3. Public vs internal field classification

In [ ]:
cls = coverage['schema_coverage'].groupby('source_class').size().rename('n_fields').reset_index()
display(cls)
display(coverage['schema_coverage'][['field','source_class','ground_truth_class','relevance_note']])


## 4. Corporate-action-type × field coverage

In [ ]:
etc = coverage['event_type_coverage']
display(etc.head(50))
etc.to_csv(OUTPUTS / 'event_type_coverage.csv', index=False)
if not etc.empty:
    pivot = etc.pivot_table(index='field', columns='corporate_action_type', values='coverage_pct')
    display(pivot)
    fig, ax = plt.subplots(figsize=(10,12))
    sns.heatmap(pivot, cmap='Blues', vmin=0, vmax=100, ax=ax)
    ax.set_title('corporate_action_type × field coverage (sample)')
    fig.tight_layout(); fig.savefig(OUTPUTS/'figures'/'event_type_coverage_heatmap.png', dpi=150); plt.show()


## 5. Document characteristics

In [ ]:
doc_stats = analyze_documents(docs, content_sample)
display(doc_stats['format_distribution'])
display(doc_stats['missing_core_fields'])
display(doc_stats['sample_status_counts'])
ok = content_sample[content_sample['status']=='ok']
if not ok.empty:
    display(ok.groupby('form')[['words','tokens_whitespace','pages_approx_500w','n_html_tables','table_word_share']].mean(numeric_only=True).round(2))
print(doc_stats['note'])
fig = plot_format_distribution(docs); plt.show()


## 6. Temporal / amendment analysis

In [ ]:
amend = analyze_amendments(docs, events, content_sample)
print(f"Events with amendments: {amend['n_events_with_amendments']} ({amend['pct_events_with_amendments']}%)")
display(amend['simple_amendment_examples'])
display(amend['complex_amendment_examples'])
display(amend['field_presence_delta_summary'])
print(amend['note'])
print('Lifecycle model: initial state → amendment(s) → final/completed/cancelled (do not overwrite prior values).')


## 7. Missingness analysis

In [ ]:
miss = classify_missingness(coverage['schema_coverage'], coverage['event_type_coverage'])
display(miss)
miss.to_csv(OUTPUTS / 'missingness_taxonomy.csv', index=False)


## 8. Ground-truth feasibility

In [ ]:
gt = ground_truth_recommendations(coverage['schema_coverage'])
display(gt)
gt.to_csv(OUTPUTS / 'ground_truth_feasibility.csv', index=False)


## 9. Dataset splitting and leakage

In [ ]:
quality = analyze_data_quality(docs, events)
display(quality['leakage_risks'])
display(quality['missing_metadata'])
mvp = recommend_mvp_design(docs, events)
display(mvp['split_table'])
print(mvp['split_policy'])


## 10. Implications for MVP Design

See `EDA_SUMMARY.md` for the written recommendations answering:
1. Which corporate-action types to start with
2. Which BNY fields can be automated from public data
3. Which fields need BNY/internal data
4. Which need manual annotation
5. Best amendment examples
6. First extraction benchmark field subset
7. Sufficiency for extraction / notification generation / amendment detection / grounding


In [ ]:
# Persist required reports
quality['missing_metadata'].assign(section='missing_metadata').to_csv(OUTPUTS/'data_quality_report.csv', index=False)
print('Primary outputs directory:', OUTPUTS)
print('Or re-run: python scripts/run_eda.py')
